In [ ]:
"""
==============================================================================
🚀 [Step 5] PyTorch Dataset Integration & YOLO Converter
==============================================================================
### @Author       : 한의정 (Data Engineering Lead)
### @Description  : 최종 가공된 전처리 데이터를 모델링 학습 루프(Train Loop)에 통합.
###                 Faster R-CNN / RetinaNet용 DataLoader 및 YOLO용 레이블 변환기 제공.
### @Bridge       : Modeling Team 배포용 최종 표준 데이터셋 (COCO Format 기반)
### @Input        : train_letterbox.json, val_letterbox.json, letterbox_images/
### @Output       : DataLoader 검증 완료, yolo_labels/, data.yaml
==============================================================================
"""


## 📊 Data Constraint Report — 희귀 클래스(Minority) 제약 사항

| 항목 | 내용 |
|---|---|
| **현상** | 데이터 1장 보유 희귀 클래스 6종 → Train 셋 전량 배정 |
| **영향** | Val 셋에 해당 클래스 없어 mAP 0으로 표시될 수 있음 |
| **대응** | Copy-Paste 증강으로 학습 수행, 최종 평가는 Confusion Matrix로 보완 예정 |

> 💡 mAP가 낮게 표시되는 클래스가 있어도 학습은 정상적으로 진행됩니다.
> Kaggle 리더보드 제출이 신뢰할 수 있는 유일한 평가 방법입니다.


In [ ]:
import sys, os

try:
    is_colab = 'google.colab' in str(get_ipython())
except NameError:
    is_colab = False

if is_colab:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception:
        pass  # 이미 마운트됐거나 VSCode 커널 환경

    REPO_DIR = '/content/pill_detection_project'
    if not os.path.exists(REPO_DIR):
        os.system('git clone https://github.com/wina0901/pill_detection_project.git ' + REPO_DIR)

    # nested 폴더 대응 (pill_detection_project/pill_detection_project/src/ 구조)
    PROJECT_ROOT = REPO_DIR
    nested = os.path.join(REPO_DIR, 'pill_detection_project')
    if os.path.isdir(os.path.join(nested, 'src')):
        PROJECT_ROOT = nested

    sys.path.insert(0, PROJECT_ROOT)
    BASE_DIR = '/content/drive/MyDrive/data/초급_프로젝트/dataset'

else:
    REPO_DIR = os.path.expanduser('~/Desktop/vera_pill')
    sys.path.insert(0, REPO_DIR)
    BASE_DIR = os.path.expanduser('~/Desktop/vera_pill/dataset')

print(f"환경    : {'Colab' if is_colab else '로컬'}")
print(f"REPO_DIR: {sys.path[0]}")
print(f"BASE_DIR: {BASE_DIR}")
assert os.path.exists(BASE_DIR), f"🚨 경로 없음: {BASE_DIR}"


In [ ]:
import os, json, random, time, warnings
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from collections import defaultdict
from src.preprocessing.dataset import (
    get_loaders, validate_coco, build_df_from_json, OralDrugDataset, denormalize
)
from src.preprocessing.format_converter import run_yolo_conversion
from src.preprocessing.viz_utils import show_samples

plt.rcParams['font.family']        = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings('ignore')
print("✅ 라이브러리 로드 완료")
print(f"PyTorch: {torch.__version__}")


## 1. 경로 설정

In [ ]:
TRAIN_JSON_PATH = os.path.join(BASE_DIR, 'train_letterbox.json')
VAL_JSON_PATH   = os.path.join(BASE_DIR, 'val_letterbox.json')
TRAIN_IMG_DIR   = os.path.join(BASE_DIR, 'letterbox_images/train')
VAL_IMG_DIR     = os.path.join(BASE_DIR, 'letterbox_images/val')

assert os.path.exists(TRAIN_JSON_PATH), "NB03~04를 먼저 실행하세요"


## 2. 파이프라인 무결성 검증 (Sanity Check)

In [ ]:
def validate_coco(json_path, target_size=800):
    """
    Letterbox 전처리가 완료된 COCO JSON 파일의 무결성을 검사합니다.

    검사 항목:
      1) 이미지 크기가 모두 target_size x target_size인지
      2) BBox 좌표에 음수가 없는지 (negative_xy)
      3) BBox 크기가 양수인지 (non_positive_wh)
      4) BBox가 이미지 경계 안에 있는지 (out_of_bounds)
    """
    if not os.path.exists(json_path):
        print(f"🚨 파일을 찾을 수 없습니다: {json_path}")
        return

    with open(json_path, 'r', encoding='utf-8') as f:
        coco = json.load(f)

    # 이미지 크기 검사: Letterbox 후 모든 이미지가 800x800이어야 함
    wrong_size = [img for img in coco['images']
                  if img['width'] != target_size or img['height'] != target_size]

    # BBox 좌표 이상 여부 검사
    issues = []
    for ann in coco['annotations']:
        x, y, w, h = ann['bbox']
        if x < 0 or y < 0:
            issues.append(('negative_xy',      ann['id']))
        if w <= 0 or h <= 0:
            issues.append(('non_positive_wh',  ann['id']))
        if x + w > target_size or y + h > target_size:
            issues.append(('out_of_bounds',    ann['id']))

    print(f"\n[{os.path.basename(json_path)}]")
    print(f"  • 이미지 수        : {len(coco['images'])}장")
    print(f"  • BBox 총 수       : {len(coco['annotations'])}개")
    print(f"  • 규격 이상 이미지  : {len(wrong_size)}장  ← 0이어야 정상")
    print(f"  • BBox 좌표 이슈   : {len(issues)}개     ← 0이어야 정상")

print("=== Train ===")
validate_coco(TRAIN_JSON_PATH)
print("\n=== Val ===")
validate_coco(VAL_JSON_PATH)


## 3. DataLoader 생성

In [ ]:
train_loader, val_loader, orig2model, num_classes, val_json = get_loaders(
    base_dir    = BASE_DIR,
    batch_size  = 2,
    num_workers = 2,
)

# 역방향 매핑: 모델 레이블 → 원본 category_id (추론 결과 해석 시 사용)
model2orig = {v: k for k, v in orig2model.items()}

print(f"\nnum_classes (background 포함): {num_classes}  ← 모델 정의 시 이 값 사용")
print(f"orig2model 샘플: {dict(list(orig2model.items())[:5])}")
print(f"\n🚀 DataLoader 준비 완료 | Train {len(train_loader)} steps / Val {len(val_loader)} steps")


## 4. 배치 1개 로드 & 형태 확인

In [ ]:
images, targets = next(iter(train_loader))
print(f"배치 이미지 수: {len(images)}")
for i, (img, tgt) in enumerate(zip(images, targets)):
    print(f"  [{i}] image: {img.shape}  dtype={img.dtype}  range=[{img.min():.3f}, {img.max():.3f}]")
    print(f"       boxes:  {tgt['boxes'].shape}  labels: {tgt['labels'].tolist()}")
    print(f"       image_id: {tgt['image_id'].item()}")


## 5. 📸 배치 시각화 (BBox 포함)

In [ ]:
def viz_batch(images, targets, model2orig, cat_json_path=None, n=4):
    """배치 이미지를 BBox와 함께 시각화합니다."""
    cat_names = {}
    if cat_json_path and os.path.exists(cat_json_path):
        with open(cat_json_path) as f:
            coco = json.load(f)
        cat_names = {c['id']: c['name'] for c in coco['categories']}

    n = min(n, len(images))
    fig, axes = plt.subplots(1, n, figsize=(n * 5, 5))
    if n == 1: axes = [axes]
    colors = plt.cm.Set1.colors

    for i in range(n):
        img    = images[i]
        tgt    = targets[i]
        img_np = img.permute(1, 2, 0).numpy()
        axes[i].imshow(img_np)

        for box, label in zip(tgt['boxes'], tgt['labels']):
            x1, y1, x2, y2 = box.tolist()
            w, h      = x2 - x1, y2 - y1
            orig_cat  = model2orig.get(label.item(), label.item())
            name      = cat_names.get(orig_cat, str(orig_cat))
            color     = colors[label.item() % len(colors)]
            rect      = patches.Rectangle((x1, y1), w, h, lw=1.5,
                                          edgecolor=color, facecolor='none')
            axes[i].add_patch(rect)
            axes[i].text(x1, y1-2, name[:10], fontsize=6, color='white',
                         bbox=dict(fc=color, alpha=0.7, pad=0.1))
        axes[i].set_title(f"image_id={tgt['image_id'].item()}", fontsize=8)
        axes[i].axis('off')

    plt.suptitle("DataLoader 배치 시각화", fontsize=13)
    plt.tight_layout(); plt.show()

viz_batch(images, targets, model2orig, TRAIN_JSON_PATH, n=min(2, len(images)))


## 6. 📸 Train 전처리 최종 결과 20장

In [ ]:
show_samples(TRAIN_IMG_DIR, TRAIN_JSON_PATH, n=20,
             title="최종 Train 데이터 (Letterbox + CLAHE + CopyPaste)")


## 7. 📸 Val 최종 결과 20장

In [ ]:
show_samples(VAL_IMG_DIR, VAL_JSON_PATH, n=20,
             title="최종 Val 데이터 (Letterbox + CLAHE)")


## 8. DataLoader 속도 벤치마크

In [ ]:
print("Train DataLoader 속도 벤치마크 (3 배치)...")
start = time.time()
for i, (imgs, tgts) in enumerate(train_loader):
    if i >= 3: break
elapsed = time.time() - start
print(f"3 배치 로딩 시간: {elapsed:.2f}s  ({elapsed/3:.2f}s/배치)")
print(f"배치당 이미지: {len(imgs)}장  →  처리율: {len(imgs)*3/elapsed:.1f}장/s")


## 9. 🚀 YOLO 포맷 변환 및 data.yaml 생성

In [ ]:
# COCO JSON → YOLO txt 변환 + data.yaml 자동 생성
# 🚨 YOLO 레이블은 0-based (Faster R-CNN의 orig2model과 1 차이)
# train/val 모두 동일한 cat2yolo 매핑을 사용해야 클래스 ID가 일치합니다.
run_yolo_conversion(BASE_DIR)

import glob as _glob
txt_files = _glob.glob(os.path.join(BASE_DIR, 'yolo_labels', 'train', '*.txt'))
print(f"\n생성된 YOLO 라벨: {len(txt_files)}개")


## ✅ 전처리 파이프라인 완료

| 단계 | 내용 | 출력 |
|------|------|------|
| NB01 | EDA | 분포 확인 |
| NB02 | Stratified Split + Copy-Paste | train_augmented_final.json |
| NB03 | Letterbox 800×800 | letterbox_images/ |
| NB04 | CLAHE 대비 강화 | in-place |
| NB05 | DataLoader 검증 + YOLO 변환 | ✅ |

> 다음 단계: 모델 학습 (Faster R-CNN / RetinaNet / YOLO)
